# Cat Behaviour Video Classifier

Create, train and export a cat behaviour classifier from labelled track embeddings.

## Configure notebook

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import base64
import json
from dataclasses import InitVar, dataclass, field
from enum import IntEnum

import cv2
import ipywidgets as widgets
import joblib
import numpy as np
from IPython.display import display

from utils import get_video_hashes

In [ ]:
# Define constants
PANEL_WIDTH = 480

In [ ]:
# Define paths
DATA_ROOT = PROJECT_ROOT / "datasets" / "behaviour_dataset"
ANN_PATH = DATA_ROOT / "behaviour_labels.json"
TRACK_FILES = sorted(DATA_ROOT.glob("track-*.joblib"))

## Load and validate tracks

In [ ]:
class AnnotationStatus(IntEnum):
    COMPLETE = 0
    VIDEO_UNLABELLED = 1
    VIDEO_UNLABELLED_CAT = 2
    PARTIAL = 3
    OVERLAPPING = 4


@dataclass
class TrackInfo:
    video_path: str
    cat_id: str
    frame_hashes: list[str]
    images: dict[str, np.ndarray]
    embeddings: dict[str, np.ndarray]
    embedding_model_sha256: str
    annotations: InitVar[list[dict]]
    video_name: str = field(init=False)
    annotation_status: AnnotationStatus = field(init=False)
    behaviour_map: dict[str, str] | None = field(init=False, default=None)

    def __post_init__(self, annotations: list[dict]) -> None:
        self.video_name = Path(self.video_path).name
        self.annotation_status = self._join_annotations(annotations)

    def __len__(self) -> int:
        return len(self.frame_hashes)

    def _join_annotations(self, annotations: list[dict]) -> AnnotationStatus:
        # check if the video is unlabelled
        video_annotations = [
            a for a in annotations if a["video_name"] == self.video_name
        ]
        if not video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED

        # check for cat specific annotations
        cat_video_annotations = [
            ann for ann in video_annotations if ann["cat_id"] == self.cat_id
        ]
        if not cat_video_annotations:
            return AnnotationStatus.VIDEO_UNLABELLED_CAT

        # create hash map
        hash_map = {h: i for i, h in enumerate(get_video_hashes(self.video_path))}

        # convert annotations to index based windows
        windows = []
        for ann in cat_video_annotations:
            start = hash_map[ann["frame_hash_start"]]
            end = hash_map[ann["frame_hash_end"]]
            windows.append((start, end, ann["behaviour"]))

        # create behaviour map for each track frame
        behaviour_map = {}
        for frame_hash in self.frame_hashes:
            matches = [b for s, e, b in windows if s <= hash_map[frame_hash] <= e]
            if len(matches) > 1:
                return AnnotationStatus.OVERLAPPING
            if not matches:
                return AnnotationStatus.PARTIAL
            behaviour_map[frame_hash] = matches[0]

        # set the behaviour map
        self.behaviour_map = behaviour_map
        return AnnotationStatus.COMPLETE


# load annotations and track information
annotations = json.loads(ANN_PATH.read_text()) if ANN_PATH.exists() else []
all_tracks = [TrackInfo(**joblib.load(p), annotations=annotations) for p in TRACK_FILES]

In [ ]:
# categorise tracks based status
complete_tracks = [
    t for t in all_tracks if t.annotation_status == AnnotationStatus.COMPLETE
]
incorrectly_labelled_tracks = [
    t
    for t in all_tracks
    if t.annotation_status
    in [
        AnnotationStatus.VIDEO_UNLABELLED_CAT,
        AnnotationStatus.PARTIAL,
        AnnotationStatus.OVERLAPPING,
    ]
]
unlabelled_videos = {
    t.video_name
    for t in all_tracks
    if t.annotation_status == AnnotationStatus.VIDEO_UNLABELLED
}
print(f"Complete tracks: {len(complete_tracks)}")
print(f"Incorrectly labelled tracks: {len(incorrectly_labelled_tracks)}")
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")

## Inspect invalid tracks

In [ ]:
current_video = {"path": None, "images": [], "hashes": [], "height": None}


def image_to_html(image: np.ndarray, height: int | None = None) -> str:
    success, encoded = cv2.imencode(".jpg", image)
    if not success:
        raise RuntimeError("OpenCV could not encode the preview frame")
    data = base64.b64encode(encoded.tobytes()).decode("ascii")
    size = (
        f"width:100%;height:{height}px;object-fit:contain"
        if height
        else "width:100%;height:auto"
    )
    return f'<img src="data:image/jpeg;base64,{data}" style="display:block;{size}">'


def load_video(track: TrackInfo) -> None:
    if current_video["path"] == track.video_path:
        return
    capture = cv2.VideoCapture(track.video_path)
    images = []
    while True:
        success, image = capture.read()
        if not success:
            break
        images.append(image)
    capture.release()
    hashes = list(get_video_hashes(track.video_path))
    height = None
    if images:
        h, w = images[0].shape[:2]
        height = round(PANEL_WIDTH * h / w)
    current_video.update(
        path=track.video_path, images=images, hashes=hashes, height=height
    )


def make_panel() -> tuple[widgets.HTML, widgets.IntSlider, widgets.HTML]:
    view = widgets.HTML(
        layout=widgets.Layout(width=f"{PANEL_WIDTH}px", border="1px solid #555")
    )
    slider = widgets.IntSlider(
        description="Index",
        min=0,
        max=0,
        value=0,
        continuous_update=True,
        layout=widgets.Layout(width=f"{PANEL_WIDTH}px"),
    )
    return view, slider, widgets.HTML()


track_dropdown = widgets.Dropdown(
    options=[
        (f"{t.video_name} | {t.cat_id} | {t.annotation_status.name}", i)
        for i, t in enumerate(incorrectly_labelled_tracks)
    ],
    description="Track",
    layout=widgets.Layout(width=f"{PANEL_WIDTH * 2}px"),
)
video_view, video_slider, video_status = make_panel()
track_view, track_slider, track_status = make_panel()


def refresh_video(*_) -> None:
    images, hashes = current_video["images"], current_video["hashes"]
    if not images:
        video_view.value = ""
        video_status.value = "<b>Raw video</b> &nbsp; no decodable frames"
        return
    index = video_slider.value
    frame_hash = hashes[index] if index < len(hashes) else "-"
    video_status.value = (
        f"<b>Raw video</b> &nbsp; index {index}/{len(images) - 1}"
        f" &nbsp; hash: {frame_hash}"
    )
    video_view.value = image_to_html(images[index])


def refresh_track(*_) -> None:
    track = incorrectly_labelled_tracks[track_dropdown.value]
    index = track_slider.value
    frame_hash = track.frame_hashes[index]
    track_status.value = (
        f"<b>Track</b> &nbsp; index {index}/{len(track) - 1}"
        f" &nbsp; hash: {frame_hash}"
    )
    image = track.images.get(frame_hash)
    height = current_video["height"]
    if image is not None:
        track_view.value = image_to_html(image, height=height)
    else:
        track_view.value = f'<div style="height:{height or 0}px"></div>'


def on_track_change(*_) -> None:
    track = incorrectly_labelled_tracks[track_dropdown.value]
    load_video(track)
    video_slider.value = 0
    video_slider.max = max(len(current_video["images"]) - 1, 0)
    track_slider.value = 0
    track_slider.max = max(len(track) - 1, 0)
    refresh_video()
    refresh_track()


track_dropdown.observe(on_track_change, names="value")
video_slider.observe(refresh_video, names="value")
track_slider.observe(refresh_track, names="value")

In [ ]:
if incorrectly_labelled_tracks:
    display(
        widgets.VBox(
            [
                track_dropdown,
                widgets.HBox(
                    [
                        widgets.VBox([video_slider, video_status, video_view]),
                        widgets.VBox([track_slider, track_status, track_view]),
                    ]
                ),
            ]
        )
    )
    on_track_change()
else:
    print("No partial or invalid tracks to inspect.")